# Trace — Job Matching Model

**Two-stage retrieval:**
Multilingual bi-encoder (`paraphrase-multilingual-mpnet-base-v2`) → cosine similarity (Stage 1) → heuristic weighted reranker (Stage 2)

**Three eval cases:**
1. Happy path — welder near Lekki
2. Multilingual retrieval — Pidgin generator post, Yaba
3. Rate + geography reranking — domestic worker, Lekki

All outputs are deterministic (seed=42, embeddings cached to disk).

In [ ]:
# Run once to install dependencies — skip if already installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'sentence-transformers', 'torch', '--quiet'])

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from training.job_match_synthetic import load_jobs, generate_workers, TEST_CASES
from training.job_match_engine import JobMatchEngine, WEIGHTS

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Load data

In [1]:
jobs    = load_jobs()
workers = generate_workers(n=200, seed=42)

print(f"Jobs:    {len(jobs)}  |  languages: {jobs['posted_language'].value_counts().to_dict()}")
print(f"Workers: {len(workers)}")
print(f"Cold-start (0 gigs): {(workers['completed_gigs'] == 0).sum()} "
      f"({100*(workers['completed_gigs']==0).mean():.0f}%)")
print()
print("Worker location distribution (top 8):")
print(workers['location_name'].value_counts().head(8).to_string())

# Persist to disk — commit these files and never regenerate on demo day
from training.job_match_synthetic import save_to_json
save_to_json(output_dir='../data') 

NameError: name 'load_jobs' is not defined

## 2. Initialise engine + embed workers
First run: ~30s to encode 200 profiles. Every subsequent run: <1s from cache.

In [ ]:
engine = JobMatchEngine(model_name='paraphrase-multilingual-mpnet-base-v2')
engine.load_workers(workers, cache_path='../models/worker_embeddings.npy')
print("Engine ready.")

## 3. Multilingual embedding sanity check
Verify the model produces meaningful cross-lingual embeddings before relying on them in the demo.
- Hausa tailoring ↔ English tailoring → **HIGH** (same domain, different language)
- Hausa tailoring ↔ English welding → **LOW** (different domain entirely)

In [ ]:
import numpy as np

def _embed_one(text):
    raw = engine.model.encode([text], convert_to_numpy=True)
    v = raw / np.linalg.norm(raw)
    return v[0]

hausa_tailoring   = jobs[jobs['job_id'] == 'job_011'].iloc[0]
english_tailoring = jobs[jobs['job_id'] == 'job_009'].iloc[0]
english_welding   = jobs[jobs['job_id'] == 'job_001'].iloc[0]
yoruba_painting   = jobs[jobs['job_id'] == 'job_032'].iloc[0]

e_ha   = _embed_one(f"{hausa_tailoring['title']}. {hausa_tailoring['description']}")
e_en_t = _embed_one(f"{english_tailoring['title']}. {english_tailoring['description']}")
e_en_w = _embed_one(f"{english_welding['title']}. {english_welding['description']}")
e_yo   = _embed_one(f"{yoruba_painting['title']}. {yoruba_painting['description']}")

checks = [
    ('Hausa tailoring ↔ English tailoring', float(e_ha @ e_en_t),  'HIGH', lambda s: s > 0.55),
    ('Hausa tailoring ↔ English welding  ', float(e_ha @ e_en_w),  'LOW',  lambda s: s < 0.50),
    ('Yoruba painting  ↔ English tailoring', float(e_yo @ e_en_t), 'MED',  lambda s: 0.30 < s < 0.80),
    ('Yoruba painting  ↔ English welding  ', float(e_yo @ e_en_w), 'LOW',  lambda s: s < 0.55),
]

all_pass = True
for label, score, expectation, test in checks:
    ok = test(score)
    all_pass = all_pass and ok
    print(f"{'✓' if ok else '✗ FAIL'}  {label}  (expect {expectation})  →  {score:.3f}")

print()
print("Multilingual embeddings look good — safe to demo." if all_pass
      else "⚠  One or more checks failed — investigate before the pitch.")

## 4. Eval case 1 — Happy path (welder near Lekki)

In [ ]:
hero = TEST_CASES[0]
job  = jobs[jobs['job_id'] == hero['job_id']].iloc[0].to_dict()

print(f"Job:         {job['title']}")
print(f"Location:    {job['location_name']}")
print(f"Budget:      ₦{job['budget_naira']:,}/day")
print(f"Language:    {job['posted_language']}")
print(f"\nDescription: {job['description']}")
print(f"\nExpected: {hero['description']}")

In [ ]:
results1 = engine.match(job, top_k=5)

display_cols = [
    'name', 'location_name', 'distance_km', 'daily_rate_naira',
    'completed_gigs', 'kudiscore_tier',
    'semantic_score', 'location_score', 'performance_score', 'rate_score', 'final_score',
]
display_cols = [c for c in display_cols if c in results1.columns]

(results1[display_cols]
    .assign(daily_rate_naira=results1['daily_rate_naira'].apply(lambda x: f'₦{x:,}'))
    .rename(index=lambda i: i + 1))

## 5. Eval case 2 — Multilingual retrieval (Pidgin generator post, Yaba)

In [ ]:
hero2 = TEST_CASES[1]
job2  = jobs[jobs['job_id'] == hero2['job_id']].iloc[0].to_dict()

print(f"Job (Pidgin): {job2['title']}")
print(f"Description:  {job2['description']}")
print(f"\nExpected: {hero2['description']}")
print()

results2 = engine.match(job2, top_k=5)
display_cols2 = [c for c in display_cols if c in results2.columns]
(results2[display_cols2]
    .assign(daily_rate_naira=results2['daily_rate_naira'].apply(lambda x: f'₦{x:,}'))
    .rename(index=lambda i: i + 1))

## 6. Eval case 3 — Rate + geography reranking (domestic worker, Lekki)

In [ ]:
hero3 = TEST_CASES[2]
job3  = jobs[jobs['job_id'] == hero3['job_id']].iloc[0].to_dict()

print(f"Job:         {job3['title']}")
print(f"Budget:      ₦{job3['budget_naira']:,}/day")
print(f"Description: {job3['description']}")
print(f"\nExpected: {hero3['description']}")
print()

results3 = engine.match(job3, top_k=5)
display_cols3 = [c for c in display_cols if c in results3.columns]
(results3[display_cols3]
    .assign(daily_rate_naira=results3['daily_rate_naira'].apply(lambda x: f'₦{x:,}'))
    .rename(index=lambda i: i + 1))

## 7. Score breakdown chart — Eval case 1 (Welder, Lekki)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

names       = results1['name'].tolist()
score_cols  = ['semantic_score', 'location_score', 'performance_score', 'rate_score']
weight_keys = ['semantic', 'location', 'performance', 'rate']
colors      = ['#2563EB', '#16A34A', '#D97706', '#DC2626']
labels      = ['Semantic (50%)', 'Location (20%)', 'Performance (15%)', 'Rate fit (15%)']

x      = np.arange(len(names))
bottom = np.zeros(len(names))

for col, wkey, color, label in zip(score_cols, weight_keys, colors, labels):
    if col not in results1.columns:
        continue
    contribution = results1[col].values * WEIGHTS[wkey]
    ax.bar(x, contribution, bottom=bottom, color=color, label=label, width=0.55)
    bottom += contribution

ax.plot(x, results1['final_score'].values, 'o--', color='#111827',
        linewidth=1.5, markersize=7, label='Final score', zorder=5)

ax.set_xticks(x)
ax.set_xticklabels([f"#{i+1}  {n}" for i, n in enumerate(names)], fontsize=11)
ax.set_ylabel('Weighted score contribution', fontsize=11)
ax.set_title(
    f'Top-5 workers — {job["title"]}\n{job["location_name"]}  |  Budget ₦{job["budget_naira"]:,}/day',
    fontsize=12, fontweight='bold'
)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../data/job_match.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → data/job_match.png")